# GreenMind AI: Notebook 5, a leaf detector for the hybrid disease mode + app model bundle

**Settings:** Accelerator = **GPU T4 x2**, Internet = **On**.
**Inputs (Add Input → Notebook):** `GreenMind AI: Notebook 4` (disease models) and `02_train_species_models` (species model).
Then **Save Version → Save & Run All (Commit)**, about 1–1.5 h.

1. Converts the PlantDoc **object-detection** dataset (leaf bounding boxes) to a 1-class YOLO dataset, dropping any
   training image identical to a PlantDoc classification-test image.
2. Trains a YOLOv8s **leaf detector** and reports its mAP on the PlantDoc detection test split.
3. Re-runs the leaf-crop experiment of Notebook 4 with this detector (lab-trained and field-trained EfficientNet-B0).
4. Collects the files the GreenMind app needs into `/kaggle/working/app_models/` (download this folder).

In [ ]:
import os, subprocess, sys
os.chdir("/kaggle/working")
if not os.path.exists("greenmind-ai"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/mubashir20023/greenmind-ai"], check=True)
os.chdir("/kaggle/working/greenmind-ai")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm==1.0.19", "scikit-learn", "ultralytics==8.3.187"], check=False)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
for repo in ["PlantDoc-Dataset", "PlantDoc-Object-Detection-Dataset"]:
    if not os.path.exists(f"/kaggle/working/{repo}"):
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/pratikkayal/{repo}", f"/kaggle/working/{repo}"], check=True)

In [ ]:
import glob, os
def one(pattern):
    hits = sorted(glob.glob(pattern, recursive=True))
    print(pattern, "->", hits[:1])
    return hits[0] if hits else None
D_LAB   = os.path.dirname(one("/kaggle/input/**/d_effb0_s1/best.pth") or "")
D_FIELD = os.path.dirname(one("/kaggle/input/**/d_effb0_mixed_s1/best.pth") or "")
D_SPEC  = os.path.dirname(one("/kaggle/input/**/species_effb0/best.pth") or "")
assert D_LAB and D_FIELD, "Add 'GreenMind AI: Notebook 4' as an input"
PD_TEST = "/kaggle/working/PlantDoc-Dataset/test"
LM = "tools/plantdoc_label_map.json"

In [ ]:
!python tools/plantdoc_leaf_yolo.py --od_root /kaggle/working/PlantDoc-Object-Detection-Dataset \
    --exclude_dir {PD_TEST} --out /kaggle/working/leafdet

In [ ]:
from ultralytics import YOLO
det = YOLO("yolov8s.pt")
det.train(data="/kaggle/working/leafdet/leaf.yaml", epochs=60, imgsz=640, batch=32, patience=15,
          project="/kaggle/working/yolo", name="leaf", exist_ok=True, seed=42, verbose=False, plots=True)
W = "/kaggle/working/yolo/leaf/weights/best.pt"
m = YOLO(W).val(data="/kaggle/working/leafdet/leaf.yaml", split="test", imgsz=640, verbose=False,
                project="/kaggle/working/yolo", name="leaf_test", exist_ok=True)
import json
det_metrics = {"test_mAP50": float(m.box.map50), "test_mAP50_95": float(m.box.map),
               "test_precision": float(m.box.mp), "test_recall": float(m.box.mr)}
json.dump(det_metrics, open("/kaggle/working/yolo/leaf_test_metrics.json", "w"), indent=2)
print(det_metrics)

In [ ]:
# Leaf-crop experiment with the new detector
for conf in [0.25, 0.10]:
    out = f"/kaggle/working/pd_test_leafcrops_c{int(conf*100)}"
    !python tools/disease_extras.py yolo-crops --weights {W} --src "{PD_TEST}" --label_map {LM} --conf {conf} --out {out}
    for name, d in [("lab", D_LAB), ("field", D_FIELD)]:
        !python tools/evaluate.py --ckpt {d}/best.pth --arch efficientnet_b0 --norm half --resize_mode square \
            --classes {d}/classes.json --imagefolder {out} --label_map {LM} --no_latency \
            --out /kaggle/working/results/leafcrop_c{int(conf*100)}_{name}

In [ ]:
# Files for the GreenMind app (download /kaggle/working/app_models)
import shutil, json, os
A = "/kaggle/working/app_models"
os.makedirs(A, exist_ok=True)
shutil.copy(f"{D_FIELD}/best.pth", f"{A}/leaf_health_effb0_field.pth")
shutil.copy(f"{D_FIELD}/classes.json", f"{A}/classes.json")
shutil.copy(W, f"{A}/yolo_leaf.pt")
if D_SPEC:
    shutil.copy(f"{D_SPEC}/best.pth", f"{A}/plant_id_effb0_best.pth")
    shutil.copy(f"{D_SPEC}/idx_to_species.json", f"{A}/idx_to_species_effb0.json")
else:
    print("species model not attached: add '02_train_species_models' as input to include it")
for f in sorted(os.listdir(A)):
    print(f, round(os.path.getsize(f"{A}/{f}") / 1e6, 1), "MB")

In [ ]:
import glob, json
print(open("/kaggle/working/leafdet/leafdet_stats.json").read())
print(open("/kaggle/working/yolo/leaf_test_metrics.json").read())
for f in sorted(glob.glob("/kaggle/working/pd_test_leafcrops_c*/crop_stats.json")):
    print("==", f); print(open(f).read())
for f in sorted(glob.glob("/kaggle/working/results/*/metrics.json")):
    m = json.load(open(f))
    print("==", f, "| n", m["n_images"], "| top1 %.4f" % m["top1"], "| CI", [round(x, 4) for x in m["top1_ci95"]],
          "| top5 %.4f" % m.get("top5", float("nan")), "| macroF1 %.4f" % m["macro"]["f1"], "| ECE %.4f" % m["ece_15bins"])